In [1]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser

/Users/pratyushsingh/Documents/projects/ai-impl/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = ChatGroq(model="llama-3.1-8b-instant")

In [4]:
parser = StrOutputParser()

In [5]:
res = model.invoke("Hello my name is Pratyush and yours ?")

In [8]:
res = model.invoke("can u tell my name?")

In [9]:
res

AIMessage(content="I don't have any information about your name. This is the beginning of our conversation, and I'm happy to chat with you, but I don't have any prior knowledge about you. If you'd like to share your name, I'd be happy to learn it!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 41, 'total_tokens': 97, 'completion_time': 0.078491093, 'completion_tokens_details': None, 'prompt_time': 0.001998781, 'prompt_tokens_details': None, 'queue_time': 0.159881063, 'total_time': 0.080489874}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d769e-0b43-7c61-b1ad-21f9c021049e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 56, 'total_tokens': 97})

### Including the memory is important to add in llm to remember the conversation

In [ ]:
## Normal Api is stateless memory

## ChatHistory  classes for managing chat-history

## BaseChatMessageHistory ------- Abstract class define all the methods 

## ChatMessageHistory --- Simplest In-memory version Python List

## RedisChatMessageHistory ----> Redis-db 

## FirestoreChatMessageHistory --- firestore

## streamLit - ChatMessageHistory -- storing the history in streamlit session

## RunnableWithMessageHistory -- Runnable chain history  

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate


In [11]:
message_hist = ChatMessageHistory()

In [12]:
message_hist.add_user_message("Hello")
message_hist.add_ai_message("Hi there !")
print(message_hist)

Human: Hello
AI: Hi there !


In [14]:
store = {}

In [15]:
def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]    

In [27]:
prompt = ChatPromptTemplate.from_messages([('system',"You are an Helpful AI Assistant"),('human','{input}')])
# prompt = ChatPromptTemplate(messages="")

In [28]:
chain = prompt | model | parser


In [36]:
chain_with_history = RunnableWithMessageHistory(chain,get_history,input_messages_key="input",output_messages_key="output")

In [37]:
response = chain_with_history.invoke({"input":"Hello i am pratyuhs how are you ?"},config={"configurable":{'session_id':'pratyush1234'}})

In [38]:
response

'It seems like we\'ve had a brief conversation. To clarify, you initially said:\n\n"Hello i am pratyuhs how are you ?"\n\nI responded:\n\n"Hello PratyuhS. I\'m an AI assistant, so I don\'t have feelings or emotions like humans do, but I\'m functioning properly and ready to help you with any questions or tasks you may have. How can I assist you today?"\n\nYou then asked:\n\n"What did I just say?"\n\nI replied:\n\n"You just said \'what did I just say?\' It\'s a classic self-referential question. This conversation has just started, so you didn\'t say anything prior to this. What would you like to talk about?"\n\nYou then repeated your initial message:\n\n"Hello i am pratyuhs how are you ?"\n\nIt seems like we\'ve come full circle, and I\'m ready to help you with anything you\'d like to discuss. If you have a specific question or topic in mind, feel free to ask, and I\'ll do my best to assist you.'

In [39]:
response_2 = chain_with_history.invoke({"input":"what did I just say?"},config={"configurable":{'session_id':'pratyush1234'}})

In [40]:
response_2

"You just asked 'what did I just say?' again. This conversation has been going in a loop. You initially asked 'Hello, I am pratyuhs, how are you?' and then asked a self-referential question about what you just said. After I clarified the conversation and pointed out the loop, you repeated your initial message. Now you've asked the same question again.\n\nTo break this loop and continue the conversation, let's start fresh. You can ask me a new question or discuss a topic of your interest. I'll do my best to assist you and provide helpful information."

In [44]:
print(store['pratyush1234'])

Human: Hello i am pratyuhs how are you ?
AI: Hello PratyuhS. I'm an AI assistant, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to help you with any questions or tasks you may have. How can I assist you today?
Human: what did I just say?
AI: You just said "what did I just say?" It's a classic self-referential question. This conversation has just started, so you didn't say anything prior to this. What would you like to talk about?
Human: Hello i am pratyuhs how are you ?
AI: It seems like we've had a brief conversation. To clarify, you initially said:

"Hello i am pratyuhs how are you ?"

I responded:

"Hello PratyuhS. I'm an AI assistant, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to help you with any questions or tasks you may have. How can I assist you today?"

You then asked:

"What did I just say?"

I replied:

"You just said 'what did I just say?' It's a classic self-referential question